# Сводка: время, ошибки и качество после ручной разметки

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Этот ноутбук читает только сохранённые результаты и не вызывает модели/API.
Точность считается только на метках verified. Ошибки предсказания включены в знаменатель.
Согласие нескольких моделей — исследовательский сигнал, не эталон.

In [ ]:
paths = [p for p in (ROOT/'results').glob('*.csv') if p.name != 'input_manifest.csv']
frames = [pd.read_csv(p,dtype={'image_id':str,'prediction':str},keep_default_na=False) for p in paths]
frames = [f for f in frames if {'model','image_id','prediction','variant','median_ms','error'} <= set(f.columns)]
assert frames, 'Сначала запустите 01, 02 или 03'
predictions = pd.concat(frames,ignore_index=True)
predictions['median_ms'] = pd.to_numeric(predictions.median_ms,errors='coerce')
labels = ensure_labels(image_manifest())
summary = evaluate(predictions,labels)
display(summary)
summary.to_csv(ROOT/'results/summary.csv',index=False)

In [ ]:
rgb = predictions[predictions.variant.eq('rgb')]
display(rgb.pivot_table(index='image_id',columns='model',values='prediction',aggfunc='first'))
print('Verified labels:',int(labels.label_status.eq('verified').sum()))

In [ ]:
import matplotlib.pyplot as plt
plot = summary[summary.variant.eq('rgb')].sort_values('median_ms')
fig,ax = plt.subplots(figsize=(11,max(3,len(plot)*.5)))
ax.barh(plot.model,plot.median_ms)
ax.set_xlabel('Медиана по изображениям, мс (после прогрева)')
ax.set_ylabel('Модель'); ax.set_title('Пилот: задержка на предоставленных изображениях')
fig.text(.01,.01,'Источник: results/*.csv; 9 исходных изображений; загрузка моделей исключена',fontsize=9)
plt.tight_layout(rect=(0,.05,1,1)); plt.show()

Перед выбором модели: вручную проверить метки, зафиксировать вариант preprocessing,
собрать независимые новые изображения и повторить оценку. Подбор ROI или preprocessing на тех же
9 примерах превращает их в development-набор. Проверка живого телефона и успешного прохождения
не выполняется этими офлайн-ноутбуками.